# Stanford Research Systems SIM900 and SIM960

This example connects to a SIM900 mainframe and a SIM960 analog PID controller in port 1, configures the controller, selects its display, and reads a single monitor sample.

**Hardware required:** a SIM900 with a SIM960 installed and a working VISA backend. Change the VISA address and port for your setup. The SIM960 occupies two slots; use the port connected to the module. Serial communication defaults to 9600 baud, 8 data bits, no parity, and one stop bit. If the mainframe uses another baud rate, configure it through VISA before running this example.

**Before running:** check the wiring, feedback polarity, and appropriate output limits for your experiment. Switching to manual mode or changing settings can change the physical output. This example leaves the controller in manual mode; it does not enable closed-loop feedback. Closing the VISA connection does not disable the output.

In [ ]:
from qcodes_contrib_drivers.drivers.StanfordResearchSystems.SIM900 import SIM900
from qcodes_contrib_drivers.drivers.StanfordResearchSystems.SIM960 import SIM960

sim900 = SIM900("sim900", "ASRL1::INSTR")
sim960 = SIM960(sim900, "pid", port=1)
sim900.add_submodule("pid", sim960)

## Set gain and select the display

Each setting is a QCoDeS parameter: call it with a value to set, or without arguments to query. Gain is signed; setting it also updates polarity. Selecting a display field does not change the corresponding setting.

In [ ]:
sim960.manual_output(0.0)
sim960.output_mode("manual")
sim960.gain(2.0)
sim960.display("gain")
print(sim960.gain())

In [ ]:
sim960.display("error_monitor")

## Configure the controller

Call each QCoDeS parameter with the desired value to apply its setting. Choose values appropriate to your experiment. Output limits are separate parameters and must satisfy `lower <= upper` at every step; the driver queries the opposite limit before changing one. For example, with the factory limits of -10 V and +10 V, set `sim960.output_limit_lower(-1.0)` followed by `sim960.output_limit_upper(1.0)` to narrow the range.

The controller remains in manual mode throughout configuration. Setting `output_mode` to `"pid"` would enable feedback, so only do that after checking your setup.

In [ ]:
sim960.gain(2.0)
sim960.integral(1.0)
sim960.derivative(1e-3)
sim960.offset(0.0)
sim960.polarity("positive")
sim960.ramp_rate(0.1)
sim960.ramp_enable(False)
sim960.setpoint_source("internal")
sim960.setpoint(0.0)
sim960.proportional_enable(True)
sim960.integral_enable(True)
sim960.derivative_enable(False)
sim960.offset_enable(False)

sim960.display("setpoint")

## Read a monitor and capture settings

`setpoint_monitor` reads one voltage sample, not a stream. The other read-only monitor parameters are `measure_monitor`, `error_monitor`, and `output_monitor`. Module queries flush stale input and wait for a complete response, using the mainframe's timeout. Do not use streaming commands concurrently with parameter queries or snapshots.

Registering the PID controller as a submodule includes its parameters in the station snapshot.

In [ ]:
setpoint_voltage = sim960.setpoint_monitor()
print(f"Setpoint monitor: {setpoint_voltage:.6f} V")

## Close the connection

Place the connected equipment in the desired state before closing. Only the mainframe owns a VISA connection; closing it also removes its QCoDeS submodules but does not change the physical controller settings.

In [ ]:
sim900.close()